# 01 — Preprocess all wells (batch)

Runs **both** smoothing pipelines (Savitzky-Golay + LOWESS) on every raw CSV
in `data/raw/sec/2022_02/D/`. Outputs:

- Processed CSVs in `data/processed/sec/2022_02/{savgol,lowess}/`
- Diagnostic PNGs in `results/figures/2022_02/diagnostic/`
- One row per (well × method) appended to `results/runs.csv`

Filename pattern: `{well_id}_{date}__{method_signature}.csv`

Re-run as many times as you want; each run gets its own row in the ledger,
and outputs with identical parameters overwrite themselves (same filename).


In [1]:
# Setup — make project root the working directory
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import logging

# If you opened this notebook from inside notebooks/, hop up one level.
if Path.cwd().name == "notebooks":
    os.chdir("..")

print(f"Working directory: {Path.cwd()}")
print(f"Has data/raw/sec/2022_02/D/: {Path('data/raw/sec/2022_02/D').exists()}")
print(f"Has data/metadata/wells.csv: {Path('data/metadata/wells.csv').exists()}")

Working directory: c:\Users\Mariana\Documents\karst_analysis
Has data/raw/sec/2022_02/D/: True
Has data/metadata/wells.csv: True


In [2]:
# Imports
import pandas as pd
from tqdm.notebook import tqdm

from karst_analysis.io import parse_well_filename
from karst_analysis.corrections import load_well_metadata
from karst_analysis.runs import register_run, read_ledger
from karst_analysis.sec.io import load_ysi_csv
from karst_analysis.sec.preprocessing import process_savgol, process_lowess
from karst_analysis.sec.viz import plot_diagnostic

logging.basicConfig(level=logging.WARNING, format="%(message)s")
log = logging.getLogger("preprocess")
log.setLevel(logging.INFO)

In [3]:
# Parameters — edit here for the whole run
INPUT_DIR     = Path("data/raw/sec/2022_02/D")
OUTPUT_ROOT   = Path("data/processed/sec/2022_02")
FIG_DIR       = Path("results/figures/2022_02/diagnostic")
CAMPAIGN      = "2022_02"

# Pipeline parameters
APPLY_DEPTH_ADJUSTMENT = False        # Set True if YSI/TOM offset is needed
DEPTH_ADJ_METHOD       = "TOM"
DEPTH_ADJ_VALUE        = 0.272

APPLY_LOG10 = True

# SavGol params
SAVGOL_WINDOW    = 11
SAVGOL_ORDER     = 3
SAVGOL_SEGMENTED = True

# LOWESS params
LOWESS_FRAC = 0.05
LOWESS_ITER = 2
APPLY_PAVA  = True

# Load metadata once (used to look up vadose thickness per well)
metadata = load_well_metadata()
metadata

,site,well_type,vadose_thickness_m,reference_date,source,notes
well_id,,,,,,
AW5D,AW5,D,0.640,2022-02-13,extracted from Depth from GL (m),NaN
AW6D,AW6,D,1.265,2022-02-19,extracted from Depth from GL (m),NaN
BW3D,BW3,D,3.280,2022-02-02,extracted from Depth from GL (m),NaN
LRS69D,LRS69,D,1.970,2022-02-12,extracted from Depth from GL (m),NaN
LRS70D,LRS70,D,0.830,2022-01-31,extracted from Depth from GL (m),NaN


In [4]:
# Find all raw CSVs
csvs = sorted(INPUT_DIR.glob("*.csv"))
print(f"Found {len(csvs)} CSV files:")
for c in csvs:
    print(f"  - {c.name}")

Found 5 CSV files:
  - AW5_D_YSI_20220213.csv
  - AW6_D_YSI_20220219.csv
  - BW3_D_YSI_20220202.csv
  - LRS69_D_YSI_20220212.csv
  - LRS70_D_YSI_20220131.csv


In [5]:
# Run both pipelines for every file
FIG_DIR.mkdir(parents=True, exist_ok=True)

for csv_path in tqdm(csvs, desc="Wells"):
    info = parse_well_filename(csv_path)
    df_raw = load_ysi_csv(csv_path)

    # Look up vadose thickness for this well
    vadose = float(metadata.loc[info.well_id, "vadose_thickness_m"]) \
             if info.well_id in metadata.index else None

    for method in ("savgol", "lowess"):
        if method == "savgol":
            params = {
                "method": "savgol",
                "window": SAVGOL_WINDOW, "order": SAVGOL_ORDER,
                "segmented": SAVGOL_SEGMENTED, "log10": APPLY_LOG10,
                "depth_adj": DEPTH_ADJ_METHOD if APPLY_DEPTH_ADJUSTMENT else None,
            }
            output_dir = OUTPUT_ROOT / "savgol"
        else:
            params = {
                "method": "lowess", "frac": LOWESS_FRAC, "iter": LOWESS_ITER,
                "pava": APPLY_PAVA, "log10": APPLY_LOG10,
                "depth_adj": DEPTH_ADJ_METHOD if APPLY_DEPTH_ADJUSTMENT else None,
            }
            output_dir = OUTPUT_ROOT / "lowess"

        with register_run(
            stage="smoothing",
            well_id=info.well_id, date=info.date,
            input_file=str(csv_path), params=params,
            output_dir=output_dir,
        ) as run:
            if method == "savgol":
                df_out, stats = process_savgol(
                    df_raw,
                    apply_depth_adjustment=APPLY_DEPTH_ADJUSTMENT,
                    depth_adjustment=DEPTH_ADJ_VALUE,
                    depth_adjustment_method=DEPTH_ADJ_METHOD,
                    savgol_window=SAVGOL_WINDOW, savgol_order=SAVGOL_ORDER,
                    savgol_segmented=SAVGOL_SEGMENTED,
                    apply_log10=APPLY_LOG10,
                    vadose_thickness_m=vadose,
                )
            else:
                df_out, stats = process_lowess(
                    df_raw,
                    apply_depth_adjustment=APPLY_DEPTH_ADJUSTMENT,
                    depth_adjustment=DEPTH_ADJ_VALUE,
                    depth_adjustment_method=DEPTH_ADJ_METHOD,
                    lowess_frac=LOWESS_FRAC, lowess_iter=LOWESS_ITER,
                    apply_pava=APPLY_PAVA,
                    apply_log10=APPLY_LOG10,
                    vadose_thickness_m=vadose,
                )

            df_out.to_csv(run.output_path, index=False)

            # Diagnostic PNG
            fig_path = FIG_DIR / f"{info.well_id}_{info.date}__{run.method_signature}.png"
            plot_diagnostic(
                df_raw["depth_m"].to_numpy(),  df_raw["sec_uS_cm"].to_numpy(),
                df_out["depth_m"].to_numpy(),  df_out["sec_uS_cm"].to_numpy(),
                output_path=fig_path,
                title=f"{info.well_id} {info.date} — {method}",
                method_info=run.method_signature,
            )

            run.note = ""  # add a note here if you want
            print(f"  ✓ {info.well_id} {method}  →  {run.output_path.name}")

print("\nDone.")

Wells:   0%|          | 0/5 [00:00<?, ?it/s]

  ✓ AW5D savgol  →  AW5D_20220213__savgol-w11-o3-seg.csv
  ✓ AW5D lowess  →  AW5D_20220213__lowess-f0.05-i2-pava.csv
  ✓ AW6D savgol  →  AW6D_20220219__savgol-w11-o3-seg.csv
  ✓ AW6D lowess  →  AW6D_20220219__lowess-f0.05-i2-pava.csv
  ✓ BW3D savgol  →  BW3D_20220202__savgol-w11-o3-seg.csv
  ✓ BW3D lowess  →  BW3D_20220202__lowess-f0.05-i2-pava.csv
  ✓ LRS69D savgol  →  LRS69D_20220212__savgol-w11-o3-seg.csv
  ✓ LRS69D lowess  →  LRS69D_20220212__lowess-f0.05-i2-pava.csv
  ✓ LRS70D savgol  →  LRS70D_20220131__savgol-w11-o3-seg.csv
  ✓ LRS70D lowess  →  LRS70D_20220131__lowess-f0.05-i2-pava.csv

Done.


In [6]:
# Inspect the runs ledger after the batch
ledger = read_ledger()
ledger.tail(20)

,run_id,timestamp,stage,well_id,date,method_signature,input_file,output_file,params_json,notes
2,fd37593f,2026-04-30T23:49:06,breakpoints,AW5D,20220213,bp-savgol-max10-t3,data\raw\sec\2022_02\D\AW5_D_YSI_20220213.csv,data\breakpoints\2022_02\AW5D_20220213__bp-sav...,"{""max_breakpoints"":10,""method"":""breakpoints"",""...",batch run 2026-04-30T23:53:09
3,f6d36361,2026-04-30T23:53:09,breakpoints,AW5D,20220213,bp-lowess-max10-t3,data\raw\sec\2022_02\D\AW5_D_YSI_20220213.csv,data\breakpoints\2022_02\AW5D_20220213__bp-low...,"{""max_breakpoints"":10,""method"":""breakpoints"",""...",batch run 2026-04-30T23:55:20
4,fd37593f,2026-04-30T23:55:30,breakpoints,AW6D,20220219,bp-savgol-max10-t3,data\raw\sec\2022_02\D\AW6_D_YSI_20220219.csv,data\breakpoints\2022_02\AW6D_20220219__bp-sav...,"{""max_breakpoints"":10,""method"":""breakpoints"",""...",batch run 2026-05-01T00:00:00
5,f6d36361,2026-05-01T00:00:00,breakpoints,AW6D,20220219,bp-lowess-max10-t3,data\raw\sec\2022_02\D\AW6_D_YSI_20220219.csv,data\breakpoints\2022_02\AW6D_20220219__bp-low...,"{""max_breakpoints"":10,""method"":""breakpoints"",""...",batch run 2026-05-01T00:04:08
6,fd37593f,2026-05-01T00:04:17,breakpoints,BW3D,20220202,bp-savgol-max10-t3,data\raw\sec\2022_02\D\BW3_D_YSI_20220202.csv,data\breakpoints\2022_02\BW3D_20220202__bp-sav...,"{""max_breakpoints"":10,""method"":""breakpoints"",""...",batch run 2026-05-01T00:08:16
7,f6d36361,2026-05-01T00:08:16,breakpoints,BW3D,20220202,bp-lowess-max10-t3,data\raw\sec\2022_02\D\BW3_D_YSI_20220202.csv,data\breakpoints\2022_02\BW3D_20220202__bp-low...,"{""max_breakpoints"":10,""method"":""breakpoints"",""...",batch run 2026-05-01T00:13:09
8,fd37593f,2026-05-01T00:13:18,breakpoints,LRS69D,20220212,bp-savgol-max10-t3,data\raw\sec\2022_02\D\LRS69_D_YSI_20220212.csv,data\breakpoints\2022_02\LRS69D_20220212__bp-s...,"{""max_breakpoints"":10,""method"":""breakpoints"",""...",batch run 2026-05-01T00:20:25
9,f6d36361,2026-05-01T00:20:25,breakpoints,LRS69D,20220212,bp-lowess-max10-t3,data\raw\sec\2022_02\D\LRS69_D_YSI_20220212.csv,data\breakpoints\2022_02\LRS69D_20220212__bp-l...,"{""max_breakpoints"":10,""method"":""breakpoints"",""...",batch run 2026-05-01T00:27:35
10,fd37593f,2026-05-01T00:27:43,breakpoints,LRS70D,20220131,bp-savgol-max10-t3,data\raw\sec\2022_02\D\LRS70_D_YSI_20220131.csv,data\breakpoints\2022_02\LRS70D_20220131__bp-s...,"{""max_breakpoints"":10,""method"":""breakpoints"",""...",batch run 2026-05-01T00:31:24
11,f6d36361,2026-05-01T00:31:24,breakpoints,LRS70D,20220131,bp-lowess-max10-t3,data\raw\sec\2022_02\D\LRS70_D_YSI_20220131.csv,data\breakpoints\2022_02\LRS70D_20220131__bp-l...,"{""max_breakpoints"":10,""method"":""breakpoints"",""...",batch run 2026-05-01T00:34:55
